In [ ]:
# ===========================================================================
# 1. Imports and configuration
# ===========================================================================
from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from peft import LoraConfig, get_peft_model
from sklearn.metrics import roc_auc_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from esm.models.esmc import ESMC
from esm.tokenization import EsmSequenceTokenizer
import esm.pretrained as pretrained

print("Python executable:", __import__("sys").executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# ===========================================================================
# 1. Imports and configuration (settings)
# ===========================================================================
# Paths
DATASET_PATH = Path("./data/druggability_dataset.csv")
OUTPUT_DIR = Path("./outputs")
CHECKPOINT_DIR = Path("./checkpoints/best")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Model/data settings
MODEL_PATH = "esmc_300m"       # use "esmc_600m" only if you have enough GPU memory
MAX_SEQ_LEN = 1024

# Split settings
SPLIT_SEED = 0
TRAIN_FRACTION = 0.70
DEV_FRACTION = 0.15

# Training settings; increase NUM_TRAINING_STEPS after smoke testing
NUM_TRAINING_STEPS = 1000
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 1e-4
SEED = 0
VAL_EVERY_STEPS = 250

# LoRA settings
USE_LORA = True
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.01

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ===========================================================================
# 2. Load dataset
# ===========================================================================
df = pd.read_csv(DATASET_PATH)
print(f"Total proteins: {len(df):,}")
print(f"Positive (druggable): {int(df['druggable'].sum()):,}")
print(f"Negative: {int((1 - df['druggable']).sum()):,}")
print(f"Positive rate: {df['druggable'].mean():.1%}")
df.head()

# ===========================================================================
# 2. Stratified splits
# ===========================================================================
rng = np.random.default_rng(SPLIT_SEED)

def stratified_split(df, train_frac, dev_frac, rng):
    train_idx, dev_idx, test_idx = [], [], []
    for label_value in sorted(df["druggable"].unique()):
        class_positions = df.index[df["druggable"] == label_value].to_numpy().copy()
        rng.shuffle(class_positions)
        n = len(class_positions)
        n_train = int(round(n * train_frac))
        n_dev = int(round(n * dev_frac))
        train_idx.extend(class_positions[:n_train])
        dev_idx.extend(class_positions[n_train:n_train + n_dev])
        test_idx.extend(class_positions[n_train + n_dev:])
    return sorted(train_idx), sorted(dev_idx), sorted(test_idx)

train_idx, dev_idx, test_idx = stratified_split(df, TRAIN_FRACTION, DEV_FRACTION, rng)

train_df = df.loc[train_idx].reset_index(drop=True)
dev_df = df.loc[dev_idx].reset_index(drop=True)
test_df = df.loc[test_idx].reset_index(drop=True)

print(f"Train: {len(train_df):,}  (positive rate: {train_df['druggable'].mean():.1%})")
print(f"Dev:   {len(dev_df):,}  (positive rate: {dev_df['druggable'].mean():.1%})")
print(f"Test:  {len(test_df):,}  (positive rate: {test_df['druggable'].mean():.1%})")

# ===========================================================================
# 2. Dataset class
# ===========================================================================
class ProteinDataset(Dataset):
    def __init__(self, df, sequence_col="sequence", label_col="druggable"):
        self.sequences = df[sequence_col].tolist()
        self.labels = df[label_col].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return {"sequence": self.sequences[idx], "label": self.labels[idx]}

train_dataset = ProteinDataset(train_df)
dev_dataset = ProteinDataset(dev_df)
test_dataset = ProteinDataset(test_df)

test_sequences = test_dataset.sequences
test_labels = test_dataset.labels

print(f"Train dataset: {len(train_dataset):,} sequences")
print(f"Dev dataset:   {len(dev_dataset):,} sequences")
print(f"Test dataset:  {len(test_dataset):,} sequences")

# ===========================================================================
# 3. Sequence length check
# ===========================================================================
if "length" in df.columns:
    n_truncated = int((df["length"] > MAX_SEQ_LEN).sum())
    print(f"Proteins exceeding {MAX_SEQ_LEN} residues: {n_truncated:,} ({n_truncated / len(df):.1%})")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df["length"], bins=80, alpha=0.8)
    ax.axvline(MAX_SEQ_LEN, linestyle="--", label=f"truncation = {MAX_SEQ_LEN}")
    ax.set_xlabel("Protein length (residues)")
    ax.set_ylabel("Number of proteins")
    ax.set_title("Sequence length distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No 'length' column found; skipping length plot.")

# ===========================================================================
# 4. ESMC weights download
# ===========================================================================
def prepare_esmc_300m_weights():
    """Download ESMC-300M weights and patch native ESM's data_root lookup."""
    snapshot_path = Path(snapshot_download(repo_id="biohub/esmc-300m-2024-12"))
    weights_path = snapshot_path / "data" / "weights" / "esmc_300m_2024_12_v0.pth"
    if not weights_path.exists():
        raise FileNotFoundError(f"Could not find ESMC weights at {weights_path}")

    original_data_root = pretrained.data_root

    def fixed_data_root(model: str):
        if model == "esmc-300":
            return weights_path
        return original_data_root(model)

    pretrained.data_root = fixed_data_root
    pretrained.init_empty_weights = nullcontext
    print(f"Using ESMC weights: {weights_path}")

if MODEL_PATH == "esmc_300m":
    prepare_esmc_300m_weights()
else:
    print("Using native ESMC loader for", MODEL_PATH)

# ===========================================================================
# 4. Load ESMC and attach binary classifier
# ===========================================================================
tokenizer = EsmSequenceTokenizer()
backbone = ESMC.from_pretrained(MODEL_PATH, device=device)

class ESMCForBinaryClassification(nn.Module):
    """Binary classifier wrapper around native ESMC.

    Native ESMC returns residue embeddings. This wrapper mean-pools residue
    embeddings and adds a one-logit classification head compatible with
    outputs.loss and outputs.logits style training code.
    """

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        device = next(backbone.parameters()).device
        with torch.no_grad():
            dummy_tokens = torch.ones((1, 8), dtype=torch.long, device=device)
            dummy_out = self.backbone(sequence_tokens=dummy_tokens)
            hidden_size = dummy_out.embeddings.shape[-1]
        self.classifier = nn.Linear(hidden_size, 1).to(device)

    def forward(self, input_ids=None, attention_mask=None, labels=None, sequence_tokens=None):
        if sequence_tokens is None:
            sequence_tokens = input_ids
        out = self.backbone(sequence_tokens=sequence_tokens)
        embeddings = out.embeddings

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(dtype=embeddings.dtype)
            pooled = (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled = embeddings.mean(dim=1)

        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = F.binary_cross_entropy_with_logits(logits, labels.float())
        return SimpleNamespace(loss=loss, logits=logits)

model = ESMCForBinaryClassification(backbone).to(device)
print("ESMC binary classifier loaded.")

# ===========================================================================
# 5. Attach LoRA or fall back to classifier-head training
# ===========================================================================
if USE_LORA:
    try:
        lora_config = LoraConfig(
            r=LORA_RANK,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=["out_proj"],
            modules_to_save=["classifier"],
        )
        model = get_peft_model(model, lora_config)
        print("\n=== Trainable Parameters with LoRA ===")
        model.print_trainable_parameters()
    except Exception as e:
        print("LoRA setup failed; falling back to classifier-head training only.")
        print(f"Reason: {type(e).__name__}: {e}")
        USE_LORA = False

if not USE_LORA:
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_count = sum(p.numel() for p in model.parameters())
    print("\n=== Trainable Parameters: classifier head only ===")
    print(f"trainable params: {trainable_count:,} / {total_count:,} ({100 * trainable_count / total_count:.4f}%)")

device = next(model.parameters()).device
print(f"\nDevice: {device}")

# ===========================================================================
# 6. DataLoader, evaluation, and plotting helpers
# ===========================================================================
def make_collate_fn(tokenizer, max_seq_len):
    def collate_fn(items):
        sequences = [item["sequence"] for item in items]
        labels = np.stack([item["label"] for item in items])
        tokenized = tokenizer(
            sequences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_seq_len,
        )
        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": torch.tensor(labels, dtype=torch.float32).unsqueeze(-1),
        }
    return collate_fn

collate_fn = make_collate_fn(tokenizer, MAX_SEQ_LEN)

def to_device(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}

def autocast_context(device):
    device_type = str(device).split(":")[0]
    if device_type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

@torch.no_grad()
def evaluate(model, name, dataset, device, batch_size=EVAL_BATCH_SIZE):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)
    all_probs = np.empty(len(dataset), dtype=np.float32)
    all_labels = dataset.labels.astype(np.int64)
    total_loss = 0.0
    n_seen = 0

    model.eval()
    with autocast_context(device):
        for batch in loader:
            batch = to_device(batch, device)
            outputs = model(**batch)
            bs = batch["input_ids"].size(0)
            total_loss += outputs.loss.item() * bs
            probs = torch.sigmoid(outputs.logits).squeeze(-1).float().cpu().numpy()
            all_probs[n_seen:n_seen + bs] = probs
            n_seen += bs

    loss = float(total_loss / len(dataset))
    preds = (all_probs >= 0.5).astype(np.int64)
    acc = float((preds == all_labels).mean())
    auc = float(roc_auc_score(all_labels, all_probs))
    print(f"{name}: loss={loss:.4f}  acc={acc:.4f}  auc={auc:.4f}")
    return all_probs, loss, acc, auc

def save_training_checkpoint(model, checkpoint_dir):
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    torch.save({"model_state_dict": model.state_dict(), "use_lora": USE_LORA}, checkpoint_dir / "model_state.pt")

def plot_training_metrics(losses, accuracies, val_history, window=20):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(losses, alpha=0.3, label="step")
    axes[0].plot(pd.Series(losses).rolling(window, min_periods=1).mean(), label=f"rolling mean ({window})")
    axes[0].set_xlabel("Step"); axes[0].set_ylabel("Train loss"); axes[0].set_title("Training loss"); axes[0].legend()
    axes[1].plot(accuracies, alpha=0.3, label="step")
    axes[1].plot(pd.Series(accuracies).rolling(window, min_periods=1).mean(), label=f"rolling mean ({window})")
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("Train accuracy"); axes[1].set_title("Training accuracy"); axes[1].legend()
    if val_history:
        steps = [v["step"] for v in val_history]
        aucs = [v["auc"] for v in val_history]
        axes[2].plot(steps, aucs, marker="o")
        axes[2].set_xlabel("Step"); axes[2].set_ylabel("Dev AUC"); axes[2].set_title("Dev AUC during training")
    plt.tight_layout(); plt.show()

# ===========================================================================
# 7. Train
# ===========================================================================
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    drop_last=True,
)

torch.manual_seed(SEED)
losses = []
accuracies = []
val_history = []
best_val_auc = -1.0
best_state = None

postfix = {"train_loss": "n/a", "train_acc": "n/a", "val_auc": "n/a"}
model.train()

step = 0
pbar = tqdm(total=NUM_TRAINING_STEPS, desc="training")
while step < NUM_TRAINING_STEPS:
    for batch in train_loader:
        if step >= NUM_TRAINING_STEPS:
            break
        batch = to_device(batch, device)

        with autocast_context(device):
            outputs = model(**batch)
            loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        probs = torch.sigmoid(outputs.logits).squeeze(-1).float()
        preds = (probs >= 0.5).long()
        targets = batch["labels"].squeeze(-1).long()
        acc = (preds == targets).float().mean().item()
        losses.append(loss.item())
        accuracies.append(acc)

        if (step + 1) % 20 == 0:
            postfix["train_loss"] = f"{np.mean(losses[-20:]):.3f}"
            postfix["train_acc"] = f"{np.mean(accuracies[-20:]):.2f}"
            pbar.set_postfix(postfix)

        if (step + 1) % VAL_EVERY_STEPS == 0 or step == NUM_TRAINING_STEPS - 1:
            _, val_loss, val_acc, val_auc = evaluate(model, f"dev @ step {step + 1}", dev_dataset, device)
            val_history.append({"step": step + 1, "loss": val_loss, "acc": val_acc, "auc": val_auc})
            postfix["val_auc"] = f"{val_auc:.3f}"
            pbar.set_postfix(postfix)

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                save_training_checkpoint(model, CHECKPOINT_DIR)
                print(f"  -> new best dev AUC {val_auc:.4f}; saved to {CHECKPOINT_DIR / 'model_state.pt'}")
            model.train()

        step += 1
        pbar.update(1)

pbar.close()
print(f"\nBest dev AUC during training: {best_val_auc:.4f}")

# ===========================================================================
# 7. Plot training metrics
# ===========================================================================
plot_training_metrics(losses, accuracies, val_history)

# ===========================================================================
# 8. Evaluate on held-out test set
# ===========================================================================
# For this run, evaluate the current model. It should correspond to the best checkpoint
# because the best state was saved when validation AUC improved.
checkpoint_path = CHECKPOINT_DIR / "model_state.pt"
if checkpoint_path.exists():
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state["model_state_dict"], strict=False)
    print(f"Loaded best checkpoint from {checkpoint_path}")
else:
    print("No saved checkpoint found; evaluating current in-memory model.")

model.eval()
test_probs, test_loss, test_acc, test_auc = evaluate(model, "test", test_dataset, device)

# ===========================================================================
# 8. Confusion matrix
# ===========================================================================
test_preds = (test_probs >= 0.5).astype(np.int64)
cm = confusion_matrix(test_labels.astype(np.int64), test_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["not druggable", "druggable"])
ax.set_yticklabels(["not druggable", "druggable"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Test confusion matrix\nAUC = {test_auc:.3f}  Acc = {test_acc:.3f}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

# ===========================================================================
# 9. Save test predictions
# ===========================================================================
results = pd.DataFrame({
    "uniprot_id": test_df["uniprot_id"].to_numpy() if "uniprot_id" in test_df.columns else np.arange(len(test_df)),
    "gene_symbol": test_df["gene_symbol"].to_numpy() if "gene_symbol" in test_df.columns else "",
    "sequence": test_sequences,
    "true_label": test_labels.astype(int),
    "probability_class_1": test_probs,
    "predicted_label": (test_probs >= 0.5).astype(int),
})

OUTPUT_PATH = OUTPUT_DIR / "binary_classifier_predictions.csv"
results.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(results):,} test predictions to {OUTPUT_PATH}")
results.head()
